In [ ]:
!pip install -q sentence-transformers scikit-learn

In [ ]:
!git clone https://github.com/encode/httpx.git
%cd httpx
!git checkout b5addb64f0161ff6bfe94c124ef76f6a1fba5254

fatal: destination path 'httpx' already exists and is not an empty directory.
/content/httpx
HEAD is now at b5addb6 Adapt test_response_decode_text_using_autodetect for chardet 6.0 (#3773)


In [ ]:
import os
count = 0
for root, dirs, files in os.walk("docs"):
    for f in files:
        if f.endswith(".md"):
            print(os.path.join(root, f))
            count += 1
print("Total:", count)

docs/compatibility.md
docs/index.md
docs/api.md
docs/quickstart.md
docs/environment_variables.md
docs/troubleshooting.md
docs/logging.md
docs/code_of_conduct.md
docs/contributing.md
docs/async.md
docs/exceptions.md
docs/third_party_packages.md
docs/http2.md
docs/advanced/ssl.md
docs/advanced/text-encodings.md
docs/advanced/event-hooks.md
docs/advanced/proxies.md
docs/advanced/extensions.md
docs/advanced/timeouts.md
docs/advanced/resource-limits.md
docs/advanced/clients.md
docs/advanced/transports.md
docs/advanced/authentication.md
Total: 23


In [ ]:
import re
from pathlib import Path

def encontrar_arquivos_md(pasta):
    """Retorna uma lista com os caminhos de todos os arquivos .md dentro da pasta, recursivamente."""
    return list(Path(pasta).rglob("*.md"))

def dividir_por_secoes(texto):
    """
    Divide o texto markdown em blocos, usando cabeçalhos (#, ##, ###...) como separadores.
    Retorna uma lista de tuplas (titulo_da_secao, conteudo_da_secao).
    """
    linhas = texto.split("\n")
    secoes = []
    titulo_atual = "Sem título"
    conteudo_atual = []

    for linha in linhas:
        # Detecta se a linha é um cabeçalho markdown, ex: "## Instalação"
        if re.match(r"^#{1,6}\s", linha):
            # Antes de trocar de seção, salva a anterior (se tiver conteúdo)
            if conteudo_atual:
                secoes.append((titulo_atual, "\n".join(conteudo_atual)))
            titulo_atual = linha.lstrip("#").strip()
            conteudo_atual = []
        else:
            conteudo_atual.append(linha)

    # não esquece de guardar a última seção
    if conteudo_atual:
        secoes.append((titulo_atual, "\n".join(conteudo_atual)))

    return secoes

def dividir_em_blocos(texto, tamanho_alvo=80, overlap=15):
    """
    Divide um texto longo em blocos de aproximadamente `tamanho_alvo` palavras,
    com sobreposição de `overlap` palavras entre um bloco e o próximo.
    """
    palavras = texto.split()
    if len(palavras) <= tamanho_alvo:
        return [texto.strip()] if texto.strip() else []

    blocos = []
    inicio = 0
    while inicio < len(palavras):
        fim = inicio + tamanho_alvo
        bloco = " ".join(palavras[inicio:fim])
        blocos.append(bloco)
        inicio += (tamanho_alvo - overlap)  # avança, mas "recua" um pouco (overlap)

    return blocos

In [ ]:

chunks = []       # vai guardar o texto de cada chunk
metadados = []    # vai guardar (arquivo, secao, chunk_id) de cada chunk correspondente

arquivos = encontrar_arquivos_md("docs")
chunk_id = 0

for caminho in arquivos:
    texto = caminho.read_text(encoding="utf-8")
    secoes = dividir_por_secoes(texto)

    for titulo, conteudo in secoes:
        blocos = dividir_em_blocos(conteudo)
        for bloco in blocos:
            if bloco.strip():  # ignora blocos vazios
                chunks.append(bloco)
                metadados.append({
                    "arquivo": str(caminho),
                    "secao": titulo,
                    "chunk_id": chunk_id
                })
                chunk_id += 1

print("Total de chunks criados:", len(chunks))
print("Exemplo de chunk:", chunks[0][:200])
print("Metadado correspondente:", metadados[0])

Total de chunks criados: 333
Exemplo de chunk: HTTPX aims to be broadly compatible with the `requests` API, although there are a
few design differences in places.

This documentation outlines places where the API differs...
Metadado correspondente: {'arquivo': 'docs/compatibility.md', 'secao': 'Requests Compatibility Guide', 'chunk_id': 0}


In [ ]:
from sentence_transformers import SentenceTransformer

# Carrega o modelo (na primeira vez, ele baixa os pesos da internet — pode levar 1-2 min)
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Transforma TODOS os chunks em vetores de uma vez só
# normalize_embeddings=True já deixa os vetores prontos para usar similaridade de cosseno
embeddings_chunks = modelo.encode(
    chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Formato da matriz de embeddings:", embeddings_chunks.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Formato da matriz de embeddings: (333, 384)


In [ ]:
import numpy as np

def buscar(pergunta, top_k=3):
    if not pergunta or not pergunta.strip():
        return "Erro: a pergunta está vazia. Digite uma pergunta antes de buscar."

    if len(chunks) == 0:
        return "Erro: não há documentos indexados. Verifique se o corpus foi carregado."

    if not isinstance(top_k, int) or top_k < 1:
        return f"Erro: top_k inválido ({top_k}). Use um número inteiro maior ou igual a 1."

    embedding_pergunta = modelo.encode([pergunta], normalize_embeddings=True)
    scores = np.dot(embeddings_chunks, embedding_pergunta.T).flatten()
    indices_ordenados = np.argsort(scores)[::-1][:top_k]

    resultados = []
    for posicao, idx in enumerate(indices_ordenados, start=1):
        resultados.append({
            "posicao": posicao,
            "score": float(scores[idx]),
            "trecho": chunks[idx],
            "arquivo": metadados[idx]["arquivo"],
            "secao": metadados[idx]["secao"]
        })

    return resultados

In [ ]:
def exibir_resultados(resultados):
    if isinstance(resultados, str):  # é uma mensagem de erro
        print(resultados)
        return

    for r in resultados:
        print(f"--- Resultado #{r['posicao']} (score: {r['score']:.3f}) ---")
        print(f"Arquivo: {r['arquivo']}")
        print(f"Seção: {r['secao']}")
        print(f"Trecho: {r['trecho'][:300]}...")
        print()

In [ ]:
resultados = buscar("como configurar timeout nas requisições", top_k=3)
exibir_resultados(resultados)

--- Resultado #1 (score: 0.708) ---
Arquivo: docs/advanced/timeouts.md
Seção: Setting and disabling timeouts
Trecho: You can set timeouts for an individual request:

```python...

--- Resultado #2 (score: 0.676) ---
Arquivo: docs/advanced/timeouts.md
Seção: Using a client instance:
Trecho: with httpx.Client() as client:
    client.get("http://example.com/api/v1/example", timeout=10.0)
```

Or disable timeouts for an individual request:

```python...

--- Resultado #3 (score: 0.669) ---
Arquivo: docs/advanced/timeouts.md
Seção: Setting a default timeout on a client
Trecho: You can set a timeout on a client instance, which results in the given
`timeout` being used as the default for requests made with this client:

```python
client = httpx.Client()              # Use a default 5s timeout everywhere.
client = httpx.Client(timeout=10.0)  # Use a default 10s timeout every...



Esta pergunta tem resposta objetiva e diretamente presente na documentação do HTTPX.

In [ ]:
resultados = buscar("como o httpx funciona", top_k=3)
exibir_resultados(resultados)

--- Resultado #1 (score: 0.674) ---
Arquivo: docs/advanced/proxies.md
Seção: Sem título
Trecho: HTTPX supports setting up [HTTP proxies](https://en.wikipedia.org/wiki/Proxy_server#Web_proxy_servers) via the `proxy` parameter to be passed on client initialization or top-level API functions like `httpx.get(..., proxy=...)`.

<div align="center">
    <img src="https://upload.wikimedia.org/wikiped...

--- Resultado #2 (score: 0.647) ---
Arquivo: docs/third_party_packages.md
Seção: httpx-sse
Trecho: [GitHub](https://github.com/florimondmanca/httpx-sse)

Allows consuming Server-Sent Events (SSE) with HTTPX....

--- Resultado #3 (score: 0.631) ---
Arquivo: docs/compatibility.md
Seção: Networking layer
Trecho: `requests` defers most of its HTTP networking code to the excellent [`urllib3` library](https://urllib3.readthedocs.io/en/latest/).

On the other hand, HTTPX uses [HTTPCore](https://github.com/encode/httpcore) as its core HTTP networking layer, which is a different project than `urllib3`

Esta pergunta é genérica: não aponta para uma seção específica da documentação, mas
ainda está dentro do assunto do HTTPX.

In [ ]:
resultados = buscar("qual a receita de um bolo de chocolate", top_k=3)
exibir_resultados(resultados)

--- Resultado #1 (score: 0.419) ---
Arquivo: docs/quickstart.md
Seção: Cookies
Trecho: Any cookies that are set on the response can be easily accessed: ```pycon >>> r = httpx.get('https://httpbin.org/cookies/set?chocolate=chip') >>> r.cookies['chocolate'] 'chip' ``` To include cookies in an outgoing request, use the `cookies` parameter: ```pycon >>> cookies = {"peanut": "butter"} >>> ...

--- Resultado #2 (score: 0.401) ---
Arquivo: docs/troubleshooting.md
Seção: Proxies
Trecho: ---...

--- Resultado #3 (score: 0.296) ---
Arquivo: docs/advanced/extensions.md
Seção: b"HTTP/0.9", b"HTTP/1.0", or b"HTTP/1.1"
Trecho: ```...



Esta pergunta não tem nenhuma relação com o conteúdo da documentação do HTTPX.
O objetivo é evidenciar uma limitação do núcleo de recuperação: como ele não possui
mecanismo de recusa, sempre retorna os top_k chunks mais próximos disponíveis, mesmo
quando nenhum é de fato relevante. O que diferencia esse caso é a queda nos scores
de similaridade em relação às perguntas dentro do escopo.

In [ ]:
resultado = buscar("", top_k=3)
print(resultado)

Erro: a pergunta está vazia. Digite uma pergunta antes de buscar.


In [ ]:
resultado = buscar("como configurar timeout", top_k=0)
print(resultado)

Erro: top_k inválido (0). Use um número inteiro maior ou igual a 1.


In [ ]:
pergunta_usuario = input("Digite sua pergunta sobre a documentação do HTTPX: ")
resultados = buscar(pergunta_usuario, top_k=3)
exibir_resultados(resultados)

KeyboardInterrupt: Interrupted by user